# car2asset — Cloud GPU reconstruction (Colab)

Runs the heavy 3D reconstruction step (TripoSR) on a free Colab GPU, since most laptops
(integrated graphics, no CUDA) can't run it locally at a usable speed.

**Workflow:**
1. Run all cells below.
2. Upload your `front.jpg` (TripoSR uses a single image — pick your clearest view).
3. Download the resulting `car_realistic.glb` at the end.
4. On your laptop, run:
   ```
   python -m car2asset.run --input inputs\example_car --mode both --quality fast --source-mesh path\to\car_realistic.glb
   ```
   This skips local reconstruction and uses the downloaded mesh directly, then does cartoon
   stylisation, preview rendering, and export locally (lightweight, CPU-friendly steps).

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a `T4 GPU`.

In [ ]:
!nvidia-smi

## 1. Install TripoSR

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -q -r requirements.txt
!pip install -q rembg onnxruntime

## 2. Upload your car photo

Upload the clearest single view of the car (front 3/4 angle works best for single-image reconstruction).

In [ ]:
from google.colab import files

uploaded = files.upload()
input_image_path = next(iter(uploaded.keys()))
print(f"Uploaded: {input_image_path}")

## 3. Remove background

In [ ]:
from PIL import Image
from rembg import remove

img = Image.open(input_image_path).convert("RGBA")
img_nobg = remove(img)
img_nobg.save("car_input_nobg.png")
img_nobg

## 4. Run TripoSR reconstruction

In [ ]:
!python run.py car_input_nobg.png \
    --output-dir output/ \
    --model-save-format glb \
    --mc-resolution 256 \
    --bake-texture

## 5. Rename and download the result

This produces `car_realistic.glb`, matching the filename car2asset expects.

In [ ]:
import shutil
from pathlib import Path

mesh_files = list(Path("output/0").glob("*.glb"))
assert mesh_files, "No .glb found — check the TripoSR run output above for errors."

shutil.copy(mesh_files[0], "car_realistic.glb")
files.download("car_realistic.glb")

## 6. Next step (on your laptop)

```powershell
cd C:\Users\eliss\3DCarFromImages
python -m car2asset.run `
  --input inputs\example_car `
  --mode both `
  --quality fast `
  --source-mesh C:\Users\eliss\Downloads\car_realistic.glb
```

This imports the cloud-generated mesh, then runs cartoon stylisation, preview rendering, and
GLB export locally — all of which are fast, CPU-only steps.